# Finishing Touch — Final Cleanup Pass
### NASA Li-ion Battery Dataset | CSE445.4 Machine Learning

**Purpose:** This notebook loads the already-cleaned data files and applies one final correction discovered after the initial cleanup — dropping exact zero capacity rows that were confirmed as recording failures by the NASA experiment README files.

**Input:** `data/cleaned_data/` (files from previous cleanup)  
**Output:** Overwrites `metadata_cleaned.csv` with corrected version. All other files unchanged.

---
## STEP 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import glob

print('Imports done.')

Imports done.


---
## STEP 2 — Load All Cleaned Files

In [2]:
# Metadata
df = pd.read_csv('data/cleaned_data/metadata_cleaned.csv')

# Discharge
discharge_df = pd.read_csv('data/cleaned_data/discharge_cleaned.csv')

# Impedance
impedance_df = pd.read_csv('data/cleaned_data/impedance_cleaned.csv')

# Charge — reload all chunks and merge
charge_files = sorted(glob.glob('data/cleaned_data/charge_chunks/*.csv'))
charge_df = pd.concat([pd.read_csv(f) for f in charge_files], ignore_index=True)

print('=== LOADED SHAPES ===')
print(f'metadata     : {df.shape}')
print(f'discharge_df : {discharge_df.shape}')
print(f'impedance_df : {impedance_df.shape}')
print(f'charge_df    : {charge_df.shape}')

=== LOADED SHAPES ===
metadata     : (7531, 10)
discharge_df : (764674, 11)
impedance_df : (75376, 14)
charge_df    : (6512619, 11)


---
## STEP 3 — Verify Current State of Each File

In [3]:
# ============================================================
# Confirm missing values and duplicates in all files
# ============================================================
print('=== MISSING VALUES ===')
for name, dataframe in [('metadata', df),
                         ('charge_df', charge_df),
                         ('discharge_df', discharge_df),
                         ('impedance_df', impedance_df)]:
    print(f'{name:15s}: {dataframe.isnull().sum().sum()} missing values')

print('\n=== DUPLICATES ===')
for name, dataframe in [('metadata', df),
                         ('charge_df', charge_df),
                         ('discharge_df', discharge_df),
                         ('impedance_df', impedance_df)]:
    print(f'{name:15s}: {dataframe.duplicated().sum()} duplicates')

=== MISSING VALUES ===
metadata       : 15930 missing values
charge_df      : 0 missing values
discharge_df   : 0 missing values
impedance_df   : 0 missing values

=== DUPLICATES ===
metadata       : 0 duplicates
charge_df      : 0 duplicates
discharge_df   : 0 duplicates
impedance_df   : 0 duplicates


---
## STEP 4 — Final Fix: Drop Zero Capacity Rows from Metadata

**Why:** After the initial cleanup was saved, we discovered that the NASA official README files for battery groups B0041–B0056 explicitly state:
> *"Note that there are several discharge runs where the capacity was very low. Reasons for this have not been fully analyzed."*

For group B0049–B0052 the README also states the experiment ended due to a **software crash**.

**Two categories found:**
- **Exact zeros (Capacity = 0.000)** → Recording failures → DROP
- **Non-zero but low (0 < Capacity < 0.5)** → NASA-acknowledged anomalies → KEEP

**Only metadata is affected.** discharge_df, charge_df, and impedance_df have no Capacity column.

In [4]:
# ============================================================
# Inspect zero and low capacity values before dropping
# ============================================================
discharge_meta = df[df['type'] == 'discharge']

print('=== CAPACITY DISTRIBUTION (discharge rows) ===')
print(discharge_meta['Capacity'].describe())

# Exact zeros
zero_cap = df[(df['type'] == 'discharge') & (df['Capacity'] == 0.0)]
print(f'\nExact zero rows to DROP : {len(zero_cap)}')
display(zero_cap[['battery_id', 'test_id', 'Capacity']])

# Low but non-zero
low_cap = df[(df['type'] == 'discharge') &
             (df['Capacity'] > 0.0) &
             (df['Capacity'] < 0.5)]
print(f'\nLow capacity rows to KEEP : {len(low_cap)}')
print('Batteries affected:')
print(low_cap['battery_id'].value_counts())

=== CAPACITY DISTRIBUTION (discharge rows) ===
count    2769.000000
mean        1.326543
std         0.472517
min         0.000000
25%         1.150286
50%         1.428065
75%         1.673645
max         2.640149
Name: Capacity, dtype: float64

Exact zero rows to DROP : 19


,battery_id,test_id,Capacity
50,B0047,50,0.0
132,B0047,132,0.0
164,B0047,164,0.0
234,B0045,50,0.0
348,B0045,164,0.0
418,B0048,50,0.0
500,B0048,132,0.0
532,B0048,164,0.0
602,B0046,50,0.0
684,B0046,132,0.0



Low capacity rows to KEEP : 208
Batteries affected:
battery_id
B0043    46
B0042    46
B0044    46
B0041    42
B0039    12
B0033     9
B0050     6
B0040     1
Name: count, dtype: int64


In [5]:
# ============================================================
# Drop exact zero capacity rows
# ============================================================
before = len(df)

df = df.drop(zero_cap.index).reset_index(drop=True)

after = len(df)

print(f'Rows before  : {before}')
print(f'Rows dropped : {before - after}')
print(f'Rows after   : {after}')

# Confirm no zeros remain
discharge_meta = df[df['type'] == 'discharge']
print(f'\nZero capacity rows remaining : {(discharge_meta["Capacity"] == 0.0).sum()}')
print(f'Capacity min : {discharge_meta["Capacity"].min():.6f} Ah')
print(f'Capacity max : {discharge_meta["Capacity"].max():.6f} Ah')

Rows before  : 7531
Rows dropped : 19
Rows after   : 7512

Zero capacity rows remaining : 0
Capacity min : 0.032558 Ah
Capacity max : 2.640149 Ah


---
## STEP 5 — Final Validation of All Dataframes

In [6]:
# ============================================================
# Final validation — shapes, missing values, duplicates
# ============================================================
print('=== FINAL SHAPES ===')
print(f'metadata     : {df.shape}')
print(f'charge_df    : {charge_df.shape}')
print(f'discharge_df : {discharge_df.shape}')
print(f'impedance_df : {impedance_df.shape}')

print('\n=== MISSING VALUES — FINAL ===')
for name, dataframe in [('metadata', df),
                         ('charge_df', charge_df),
                         ('discharge_df', discharge_df),
                         ('impedance_df', impedance_df)]:
    total = dataframe.isnull().sum().sum()
    note = ' <- structural NaNs by design (Capacity/Re/Rct)' if name == 'metadata' else ''
    print(f'{name:15s}: {total} missing{note}')

print('\n=== DUPLICATES — FINAL ===')
for name, dataframe in [('metadata', df),
                         ('charge_df', charge_df),
                         ('discharge_df', discharge_df),
                         ('impedance_df', impedance_df)]:
    print(f'{name:15s}: {dataframe.duplicated().sum()} duplicates')

=== FINAL SHAPES ===
metadata     : (7512, 10)
charge_df    : (6512619, 11)
discharge_df : (764674, 11)
impedance_df : (75376, 14)

=== MISSING VALUES — FINAL ===
metadata       : 15892 missing <- structural NaNs by design (Capacity/Re/Rct)
charge_df      : 0 missing
discharge_df   : 0 missing
impedance_df   : 0 missing

=== DUPLICATES — FINAL ===
metadata       : 0 duplicates
charge_df      : 0 duplicates
discharge_df   : 0 duplicates
impedance_df   : 0 duplicates


---
## STEP 6 — Overwrite Metadata File Only

Only `metadata_cleaned.csv` needs to be updated.  
All other files (discharge, impedance, charge chunks) are unchanged and do NOT need to be resaved.

In [7]:
# ============================================================
# Overwrite metadata_cleaned.csv only
# All other cleaned files are untouched
# ============================================================
df.to_csv('data/cleaned_data/metadata_cleaned.csv', index=False)

print('✅ metadata_cleaned.csv overwritten with final corrected version')
print('✅ discharge_cleaned.csv — unchanged')
print('✅ impedance_cleaned.csv — unchanged')
print('✅ charge_chunks/        — unchanged')

print('\n=== FINAL CLEANED DATA FOLDER ===')
print('data/cleaned_data/')
print('  ├── metadata_cleaned.csv    <- UPDATED')
print('  ├── discharge_cleaned.csv')
print('  ├── impedance_cleaned.csv')
print('  └── charge_chunks/')
print('       ├── charge_part_01_of_05.csv')
print('       ├── charge_part_02_of_05.csv')
print('       ├── charge_part_03_of_05.csv')
print('       ├── charge_part_04_of_05.csv')
print('       └── charge_part_05_of_05.csv')

✅ metadata_cleaned.csv overwritten with final corrected version
✅ discharge_cleaned.csv — unchanged
✅ impedance_cleaned.csv — unchanged
✅ charge_chunks/        — unchanged

=== FINAL CLEANED DATA FOLDER ===
data/cleaned_data/
  ├── metadata_cleaned.csv    <- UPDATED
  ├── discharge_cleaned.csv
  ├── impedance_cleaned.csv
  └── charge_chunks/
       ├── charge_part_01_of_05.csv
       ├── charge_part_02_of_05.csv
       ├── charge_part_03_of_05.csv
       ├── charge_part_04_of_05.csv
       └── charge_part_05_of_05.csv


---
## Summary of Changes Made in This Notebook

| Item | Action | Reason |
|------|--------|--------|
| 19 discharge rows with `Capacity = 0.0` | **Dropped** from metadata | Exact zeros are recording failures — physically impossible for a running battery |
| 208 discharge rows with `0 < Capacity < 0.5` | **Kept** | NASA-acknowledged anomalies — real readings from unusual discharge conditions |
| discharge_cleaned.csv | **Unchanged** | No Capacity column — not affected |
| impedance_cleaned.csv | **Unchanged** | No Capacity column — not affected |
| charge_chunks/ | **Unchanged** | No Capacity column — not affected |

**Data is now fully clean and ready for visualization and analysis.**